### Earthquake PINN Training on Colab GPU
This notebook clones the repository, installs dependencies, and runs the full-scale training.
It also includes optional Hyperparameter Optimization (Optuna).

**Note**: Make sure you have pushed your local changes (`git push origin dev`) before running this.

In [ ]:
print("Connected!")

In [ ]:
!nvidia-smi

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Wed Feb 18 19:16:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   62C    P0             29W /   70W |    1625MiB /  15360MiB |      0%      Default |
|                       

In [24]:
# Install uv and clean up existing clone
!pip install uv
import os

%cd /content
!rm -rf earthquake_proj
!git clone -b dev --single-branch https://github.com/sattary/earthquake_proj.git
%cd earthquake_proj

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.
/content
Cloning into 'earthquake_proj'...
remote: Enumerating objects: 431, done.
remote: Counting objects: 100% (431/431), done.
remote: Compressing objects: 100% (264/264), done.
remote: Total 431 (delta 167), reused 370 (delta 106), pack-reused 0 (from 0)
Receiving objects: 100% (431/431), 20.54 MiB | 22.37 MiB/s, done.
Resolving deltas: 100% (167/167), done.
Filtering content: 100% (32/32), 130.28 MiB | 91.75 MiB/s, done.
/content/earthquake_proj


In [ ]:
# %%bash
!git pull origin dev

From https://github.com/sattary/earthquake_proj
 * branch            dev        -> FETCH_HEAD
Already up to date.


: 

In [ ]:
# Sync environment
!uv sync

Using CPython 3.12.12 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 127 packages in 0.86ms
Installed 123 packages in 434ms                             
 + alembic==1.18.4
 + annotated-doc==0.0.4
 + asttokens==3.0.1
 + certifi==2026.1.4
 + cffi==2.0.0
 + charset-normalizer==3.4.4
 + click==8.3.1
 + cloudpickle==3.1.2
 + colorlog==6.10.1
 + comm==0.2.3
 + contourpy==1.3.3
 + coverage==7.13.4
 + cryptography==43.0.3
 + cuda-bindings==12.9.4
 + cuda-pathfinder==1.3.4
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.2.1
 + executing==2.2.1
 + filelock==3.24.0
 + fonttools==4.61.1
 + fsspec==2026.2.0
 + google-api-core==2.29.0
 + google-api-python-client==2.190.0
 + google-auth==2.48.0
 + google-auth-httplib2==0.3.0
 + googleapis-common-protos==1.72.0
 + greenlet==3.3.1
 + httplib2==0.31.2
 + idna==3.11
 + iniconfig==2.3.0
 + ipykernel==7.2.0
 + ipython==9.10.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.19.2
 + jinja2==3.1.6
 + joblib==1.5.3
 + jupyte

: 

In [ ]:
!ls

checkpoints  docs     notebooks       README.md  src	uv.lock
data	     main.py  pyproject.toml  results	 tests


: 

In [ ]:
import os

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results/tables", exist_ok=True)
os.makedirs("results/figs", exist_ok=True)

# Ensure package initialization for -m calls
if os.path.exists('src'):
    if not os.path.exists('src/__init__.py'):
        open('src/__init__.py', 'a').close()
        print("Initialized 'src' as package.")
elif os.path.exists('earthquake_proj'):
    if not os.path.exists('earthquake_proj/__init__.py'):
        open('earthquake_proj/__init__.py', 'a').close()
        print("Initialized 'earthquake_proj' as package.")

Initialized 'src' as package.


: 

In [ ]:
# --- ROBUST TRAINING PIPELINE ---
# 1. Prerequisite: Add 'github_token' to Secrets
# 2. Prerequisite: Run the 'Install & Clone' cell first

# A. Tune (Optional, remove if fixed params)
!PYTHONPATH=. uv run python -m src.pipelines.cli tune --trials 20 --epochs 500

# B. Train with RESUME + AUTO-PUSH
# --resume: Checks for checkpoints/checkpoint_epoch_X.pth and continues from latest.
# Auto-Push: Every 1000 epochs, it pushes the checkpoint to GitHub automatically.
!PYTHONPATH=. uv run python -m src.pipelines.cli train \
    --epochs 5000 \
    --n-coll 10000 \
    --config results/tables/best_params.json \
    # --resume \
    --no-multi-gpu

# C. Generate Results
!PYTHONPATH=. uv run python -m src.pipelines.cli results-suite

# D. Final Push (Results + Final Model)
# We use the utility directly to push the final artifacts
!python src/utils/persistence.py results/
!python src/utils/persistence.py checkpoints/final_model.pth

[I 2026-02-18 19:10:58,400] A new study created in memory with name: no-name-4ed6a43d-ce64-4e85-9c46-79997ec7412c
Using single device: cuda (Multi-GPU: False)
CoordinateTransformer Initialized:
  Bounds (UTM): X[-23763.9, 1477034.3] Y[2676428.4, 4376797.5]
  Scale Factor: 850184.57 meters
Loading Velocity Model from data/Morteza_2023/Vel/Pwave.3D.txt...
Building 3D Interpolator (Nearest for speed/robustness)...
Velocity Model Ready. Depth Range: 0.0-30.0 km
[I 2026-02-18 19:11:03,246] Trial 0 finished with value: 78.02008056640625 and parameters: {'f_tune': 1.5692359112223122, 'lr': 0.001381398587683496, 'w_pde': 0.1180083219994105, 'w_const': 0.13169531029712253, 'w_bc': 1.928271906237742}. Best is trial 0 with value: 78.02008056640625.

--- Tuning Complete ---
Best Trial Score: 78.0201
Saved best params to results/tables/best_params.json
Loading config from results/tables/best_params.json
[Init] Environment: Kaggle
Using single device: cuda (Multi-GPU: False)
CoordinateTransformer In

: 